# Reinforcement Learning Training of PMM Strategy Based on SAC

This notebook implements a PMM market making strategy training system based on the **Soft Actor-Critic (SAC)** algorithm.

## SAC Algorithm Features

-   **Continuous Action Space**: Suitable for parameter optimization in market making strategies
-   **Maximum Entropy Reinforcement Learning**: Maintains policy exploration while optimizing rewards
-   **High Sample Efficiency**: Uses experience replay and target networks to improve learning efficiency
-   **Good Stability**: Dual Q-network design reduces overestimation problems

## Training Objectives

-   **Strategy Optimization**: Automatically learn optimal spreads, order quantities, timing parameters, etc.
-   **Risk Control**: Control position risk while achieving returns
-   **Adaptability**: Adjust parameters to adapt to different market conditions
-   **Robustness**: Train strategies that perform stably in real markets

## ⚠️ Important: Kernel Stability Instructions

### Default Configuration (Optimized)

This notebook has been optimized for maximum stability:

-   **Device Selection**: Automatically detects CUDA, uses CPU if unavailable (avoids MPS to prevent compatibility issues)
-   **Memory Optimization**: Reduced batch_size and replay_buffer_size
-   **Test Mode**: Default training of 5 rounds for testing

### If Problems Persist

1. **Memory Shortage**
    - Reduce `batch_size` to 64
    - Reduce `replay_buffer_size` to 50000
    - Reduce `max_steps_per_episode` to 360
2. **Dependency Issues**

    ```bash
    pip install --upgrade torch tensordict hftbacktest
    ```

3. **Data Issues**
    - Ensure `02_data_preparation.ipynb` has been run
    - Check for data files in `data/output/` directory

### Recommended Workflow

1. **First Run**: Use default configuration (TEST_MODE=True, 5 training rounds)
2. **After Test Success**: Set TEST_MODE=False, increase training rounds
3. **Monitor Memory**: Watch available memory displayed in Cell 3, reduce parameters if less than 4GB

# ⚠️ Training Configuration (Set this before running other code)
# ====================================================

# Set training rounds directly (recommended approach)
NUM_EPISODES = 100         # Freely configurable: 20(test), 100(quick), 500(standard), 1000(deep)

# Other adjustable parameters
BATCH_SIZE = 64          # Batch size (reduce to 64 if memory insufficient)
REPLAY_BUFFER_SIZE = 5000  # Experience pool size (reduce to 10000 if memory insufficient)
DATA_SAMPLE_RATE = 0.1    # Data sampling rate (0.05-0.2)
SAVE_INTERVAL = 10        # Model save interval
EVAL_INTERVAL = 10        # Evaluation interval

# Display current configuration
print("=" * 50)
print("📋 SAC Training Configuration")
print("=" * 50)

print(f"📊 Training Rounds: {NUM_EPISODES:,} episodes")
print(f"💾 Batch Size: {BATCH_SIZE}")
print(f"🗄️ Experience Pool Size: {REPLAY_BUFFER_SIZE:,}")
print(f"📉 Data Sampling Rate: {DATA_SAMPLE_RATE*100:.0f}%")
print(f"💾 Save Interval: Every {SAVE_INTERVAL} rounds")
print(f"📊 Evaluation Interval: Every {EVAL_INTERVAL} rounds")

print("\n💡 Recommended Configurations:")
print("   • Test Environment: NUM_EPISODES=20")
print("   • Quick Prototype: NUM_EPISODES=100")
print("   • Standard Training: NUM_EPISODES=500")
print("   • Best Performance: NUM_EPISODES=1000")
print("=" * 50)

In [1]:
# ⚠️ Training Configuration (Set this before running other code)
# ====================================================

# Set training rounds directly (recommended approach)
NUM_EPISODES = 100         # Freely configurable: 20(test), 100(quick), 500(standard), 1000(deep)

# Other adjustable parameters
BATCH_SIZE = 64          # Batch size (reduce to 64 if memory insufficient)
REPLAY_BUFFER_SIZE = 5000  # Experience pool size (reduce to 10000 if memory insufficient)
DATA_SAMPLE_RATE = 0.1    # Data sampling rate (0.05-0.2)
SAVE_INTERVAL = 10        # Model save interval
EVAL_INTERVAL = 10        # Evaluation interval

# Display current configuration
print("=" * 50)
print("📋 SAC Training Configuration")
print("=" * 50)

print(f"📊 Training Rounds: {NUM_EPISODES:,} episodes")
print(f"💾 Batch Size: {BATCH_SIZE}")
print(f"🗄️ Experience Pool Size: {REPLAY_BUFFER_SIZE:,}")
print(f"📉 Data Sampling Rate: {DATA_SAMPLE_RATE*100:.0f}%")
print(f"💾 Save Interval: Every {SAVE_INTERVAL} rounds")
print(f"📊 Evaluation Interval: Every {EVAL_INTERVAL} rounds")

print("\n💡 Recommended Configurations:")
print("   • Test Environment: NUM_EPISODES=20")
print("   • Quick Prototype: NUM_EPISODES=100")
print("   • Standard Training: NUM_EPISODES=500")
print("   • Best Performance: NUM_EPISODES=1000")
print("=" * 50)

📋 SAC Training Configuration
📊 Training Rounds: 100 episodes
💾 Batch Size: 64
🗄️ Experience Pool Size: 5,000
📉 Data Sampling Rate: 10%
💾 Save Interval: Every 10 rounds
📊 Evaluation Interval: Every 10 rounds

💡 Recommended Configurations:
   • Test Environment: NUM_EPISODES=20
   • Quick Prototype: NUM_EPISODES=100
   • Standard Training: NUM_EPISODES=500
   • Best Performance: NUM_EPISODES=1000


In [2]:
# Environment setup and dependency imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
import warnings
import os
from collections import deque
import random
from tensordict import TensorDict
from hftbacktest import BacktestAsset
from lib.rl_env import create_pmm_env
from lib.data_slicer import DataSlicer
from tqdm.notebook import tqdm
import json
import time

# Set warning filters and random seeds
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 🔧 Global trading configuration
# Fee rate configuration
MAKER_FEE_RATE = -0.00003     # Maker fee rate -0.003% (negative fee rebate)
TAKER_FEE_RATE = 0.0007       # Taker fee rate +0.07% (positive fee charge)

# Trading specification configuration - XRP
TICK_SIZE = 0.0001            # XRP minimum price movement unit
LOT_SIZE = 0.1                # XRP minimum trading quantity unit

# Data configuration
TRAINING_PAIR = 'xrpusdt'     # XRP trading pair
START_DATE = 20250717         # Use updated data

# Display trading configuration information
print(f"📊 Global Trading Configuration:")
print(f"   Trading Pair: {TRAINING_PAIR.upper()}")
print(f"   Maker Fee Rate: {MAKER_FEE_RATE*100:.4f}% (negative fee rebate)")
print(f"   Taker Fee Rate: {TAKER_FEE_RATE*100:.4f}% (positive fee charge)")
print(f"   Minimum Price Unit: {TICK_SIZE}")
print(f"   Minimum Trading Unit: {LOT_SIZE}")

# 🔧 Device configuration (smart selection: CUDA > MPS > CPU)
def select_device():
    """Intelligently select the best available device"""
    # Priority 1: CUDA
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("🚀 Using CUDA GPU acceleration")
        print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        # Set CUDA memory management
        torch.cuda.empty_cache()
        return device
    
    # Priority 2: MPS (Apple Silicon)
    if torch.backends.mps.is_available():
        try:
            # Test if MPS is really available
            test_tensor = torch.tensor([1.0], device="mps")
            _ = test_tensor * 2
            device = torch.device("mps")
            print("🍎 Using Apple Silicon MPS acceleration")
            print("   Tip: MPS acceleration may significantly improve training speed")
            return device
        except Exception as e:
            print(f"⚠️ MPS available but initialization failed: {e}")
            print("   Falling back to CPU mode")
    
    # Priority 3: CPU
    device = torch.device("cpu")
    print("💻 Using CPU")
    print("   Tip: Training speed is slower, recommend using GPU or MPS")
    
    # Display CPU information
    import platform
    print(f"   Processor: {platform.processor()}")
    print(f"   CPU Cores: {os.cpu_count()}")
    
    return device

# Select device
device = select_device()
print(f"✅ SAC reinforcement learning environment initialization complete, using device: {device}")

📊 Global Trading Configuration:
   Trading Pair: XRPUSDT
   Maker Fee Rate: -0.0030% (negative fee rebate)
   Taker Fee Rate: 0.0700% (positive fee charge)
   Minimum Price Unit: 0.0001
   Minimum Trading Unit: 0.1
🍎 Using Apple Silicon MPS acceleration
   Tip: MPS acceleration may significantly improve training speed
✅ SAC reinforcement learning environment initialization complete, using device: mps


In [3]:
# 📊 Data Configuration
from lib.data_slicer import DataSlicer
print("📊 Initializing data system...")

# Check data file
data_file = f'data/output/{TRAINING_PAIR}_{START_DATE}.npz'
if not os.path.exists(data_file):
    raise FileNotFoundError(f"Data file does not exist: {data_file}")

print(f"✅ Data file: {os.path.basename(data_file)}")

# Use modular data slicing functionality

# Create data slices
print("📊 Preparing data slices (sliced by time)...")
slicer = DataSlicer(slices_dir='data/slices')
data_splits = slicer.split_data_by_time(
    data_file=data_file,
    pair_name=TRAINING_PAIR,
    start_date=START_DATE,
    hours_per_split=0.167  # 10 minutes per slice (10/60 hours)
)
print(f"✅ Obtained {len(data_splits)} data slices")

# Data file list for training
data_files = data_splits
print(f"✅ Data preparation complete, total {len(data_files)} training slices")

# Show information for first few slices
for i, split_file in enumerate(data_files[:3]):
    info = slicer.get_split_info(split_file)
    print(f"   Slice{i}: {info['records']:,} records, {info['duration_hours']:.1f} hours")

📊 Initializing data system...
✅ Data file: xrpusdt_20250717.npz
📊 Preparing data slices (sliced by time)...
📦 Found existing data slices (144 files), using directly...
   Segment 0: 349,899 records (0.3M) - 0.2 hours
   Segment 1: 330,220 records (0.3M) - 0.2 hours
   Segment 2: 343,814 records (0.3M) - 0.2 hours
   ... Total 144 segments
✅ Obtained 144 data slices
✅ Data preparation complete, total 144 training slices
   Slice0: 349,899 records, 0.2 hours
   Slice1: 330,220 records, 0.2 hours
   Slice2: 343,814 records, 0.2 hours


In [4]:
# 🧠 SAC Neural Network Components

# Actor Network (Policy Network)
class Actor(nn.Module):
    """SAC Actor Network - outputs action mean and log standard deviation"""

    def __init__(self, state_dim, action_dim, hidden_dim=256, log_std_min=-20, log_std_max=2):
        super(Actor, self).__init__()
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

        # Shared layers
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)

        # Mean and standard deviation output layers
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                module.bias.data.zero_()

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))

        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(
            log_std, min=self.log_std_min, max=self.log_std_max)

        return mean, log_std

    def sample(self, state):
        """Sample action and calculate log probability"""
        mean, log_std = self.forward(state)
        std = log_std.exp()

        # Create normal distribution
        normal = Normal(mean, std)
        x_t = normal.rsample()  # Reparameterization trick

        # Apply tanh transformation to limit actions to [-1, 1]
        y_t = torch.tanh(x_t)
        action = y_t

        # Calculate log probability (considering Jacobian of tanh transformation)
        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)

        mean = torch.tanh(mean)

        return action, log_prob, mean


# Critic Network (Q Network)
class Critic(nn.Module):
    """SAC Critic Network - twin Q-network architecture"""

    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(Critic, self).__init__()

        # Q1 Network
        self.q1_fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.q1_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.q1_out = nn.Linear(hidden_dim, 1)

        # Q2 Network
        self.q2_fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.q2_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.q2_out = nn.Linear(hidden_dim, 1)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                module.bias.data.zero_()

    def forward(self, state, action):
        xu = torch.cat([state, action], dim=1)

        # Q1 forward pass
        q1 = F.relu(self.q1_fc1(xu))
        q1 = F.relu(self.q1_fc2(q1))
        q1 = self.q1_out(q1)

        # Q2 forward pass
        q2 = F.relu(self.q2_fc1(xu))
        q2 = F.relu(self.q2_fc2(q2))
        q2 = self.q2_out(q2)

        return q1, q2

    def Q1(self, state, action):
        """Calculate only Q1 value"""
        xu = torch.cat([state, action], dim=1)
        q1 = F.relu(self.q1_fc1(xu))
        q1 = F.relu(self.q1_fc2(q1))
        q1 = self.q1_out(q1)
        return q1


# Experience Replay Buffer
class ReplayBuffer:
    """Experience replay buffer - stores and samples experiences"""

    def __init__(self, capacity=1000000):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Add experience to buffer"""
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        """Randomly sample a batch of experiences"""
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)


print("✅ SAC neural network components definition complete")
print(f"   - Actor Network: Policy network, outputs action distribution")
print(f"   - Critic Network: Twin Q-network, evaluates state-action value")
print(f"   - ReplayBuffer: Experience replay buffer, capacity 1 million")

✅ SAC neural network components definition complete
   - Actor Network: Policy network, outputs action distribution
   - Critic Network: Twin Q-network, evaluates state-action value
   - ReplayBuffer: Experience replay buffer, capacity 1 million


In [5]:
# 🎯 SAC Algorithm Implementation

class SAC:
    """Soft Actor-Critic algorithm implementation"""

    def __init__(
        self,
        state_dim,
        action_dim,
        action_low,
        action_high,
        device,
        lr_actor=3e-4,
        lr_critic=3e-4,
        lr_alpha=3e-4,
        gamma=0.99,
        tau=0.005,
        alpha=0.2,
        automatic_entropy_tuning=True
    ):
        self.device = device
        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha
        self.automatic_entropy_tuning = automatic_entropy_tuning

        # Action space bounds (used for scaling actions)
        self.action_low = torch.tensor(action_low, device=device)
        self.action_high = torch.tensor(action_high, device=device)
        self.action_scale = (self.action_high - self.action_low) / 2.0
        self.action_bias = (self.action_high + self.action_low) / 2.0

        # Create networks
        self.actor = Actor(state_dim, action_dim).to(device)
        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = Critic(state_dim, action_dim).to(device)

        # Copy target network parameters
        self.critic_target.load_state_dict(self.critic.state_dict())

        # Optimizers
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optimizer = optim.Adam(
            self.critic.parameters(), lr=lr_critic)

        # Automatic entropy tuning
        if self.automatic_entropy_tuning:
            self.target_entropy = - \
                torch.prod(torch.Tensor([action_dim]).to(device)).item()
            self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
            self.alpha_optimizer = optim.Adam([self.log_alpha], lr=lr_alpha)
            self.alpha = self.log_alpha.exp().item()

    def select_action(self, state, evaluate=False):
        """Select action"""
        state = torch.FloatTensor(state).to(self.device).unsqueeze(0)

        if evaluate:
            _, _, action = self.actor.sample(state)
        else:
            action, _, _ = self.actor.sample(state)

        # Scale action from [-1, 1] to actual range (fix: add detach())
        action = action.squeeze(0).detach().cpu().numpy()
        action = action * self.action_scale.cpu().numpy() + self.action_bias.cpu().numpy()

        # Ensure actions are within bounds and integers for discrete parameters
        action = np.clip(action, self.action_low.cpu().numpy(),
                         self.action_high.cpu().numpy())
        action[0] = round(action[0])  # half_spread
        action[1] = round(action[1])  # skew
        action[2] = round(action[2])  # grid_num
        action[3] = round(action[3])  # grid_interval

        return action

    def update(self, replay_buffer, batch_size=256):
        """Update network parameters"""
        if len(replay_buffer) < batch_size:
            return {}

        # Sample from experience buffer
        state, action, reward, next_state, done = replay_buffer.sample(
            batch_size)

        state = torch.FloatTensor(state).to(self.device)
        next_state = torch.FloatTensor(next_state).to(self.device)
        action = torch.FloatTensor(action).to(self.device)
        reward = torch.FloatTensor(reward).to(self.device).unsqueeze(1)
        done = torch.FloatTensor(done).to(self.device).unsqueeze(1)

        # Normalize actions to [-1, 1] (for network input)
        action_normalized = (action - self.action_bias) / self.action_scale

        with torch.no_grad():
            # Sample next action
            next_action, next_log_pi, _ = self.actor.sample(next_state)

            # Calculate target Q value
            target_q1, target_q2 = self.critic_target(next_state, next_action)
            target_q = torch.min(target_q1, target_q2) - \
                self.alpha * next_log_pi
            target_q_value = reward + (1 - done) * self.gamma * target_q

        # Update Critic
        current_q1, current_q2 = self.critic(state, action_normalized)
        critic_loss = F.mse_loss(
            current_q1, target_q_value) + F.mse_loss(current_q2, target_q_value)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # Update Actor
        new_action, log_pi, _ = self.actor.sample(state)
        q1_new, q2_new = self.critic(state, new_action)
        q_new = torch.min(q1_new, q2_new)

        actor_loss = (self.alpha * log_pi - q_new).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # Update entropy coefficient
        alpha_loss = None
        if self.automatic_entropy_tuning:
            alpha_loss = -(self.log_alpha * (log_pi +
                           self.target_entropy).detach()).mean()

            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()

            self.alpha = self.log_alpha.exp().item()

        # Soft update target network
        for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
            target_param.data.copy_(
                self.tau * param.data + (1 - self.tau) * target_param.data)

        return {
            'critic_loss': critic_loss.item(),
            'actor_loss': actor_loss.item(),
            'alpha_loss': alpha_loss.item() if alpha_loss is not None else 0,
            'alpha': self.alpha
        }

    def save(self, filepath):
        """Save model"""
        torch.save({
            'actor_state_dict': self.actor.state_dict(),
            'critic_state_dict': self.critic.state_dict(),
            'critic_target_state_dict': self.critic_target.state_dict(),
            'actor_optimizer_state_dict': self.actor_optimizer.state_dict(),
            'critic_optimizer_state_dict': self.critic_optimizer.state_dict(),
            'alpha': self.alpha,
            'log_alpha': self.log_alpha if self.automatic_entropy_tuning else None,
        }, filepath)

    def load(self, filepath):
        """Load model"""
        checkpoint = torch.load(filepath, map_location=self.device)
        self.actor.load_state_dict(checkpoint['actor_state_dict'])
        self.critic.load_state_dict(checkpoint['critic_state_dict'])
        self.critic_target.load_state_dict(
            checkpoint['critic_target_state_dict'])
        self.actor_optimizer.load_state_dict(
            checkpoint['actor_optimizer_state_dict'])
        self.critic_optimizer.load_state_dict(
            checkpoint['critic_optimizer_state_dict'])
        self.alpha = checkpoint['alpha']
        if self.automatic_entropy_tuning and checkpoint['log_alpha'] is not None:
            self.log_alpha = checkpoint['log_alpha']


print("✅ SAC algorithm implementation complete (detach issue fixed)")
print(f"   - Automatic entropy tuning: adaptive exploration-exploitation balance")
print(f"   - Twin Q-networks: reduce Q-value overestimation")
print(f"   - Soft updates: stable target network updates")
print(f"   - Reparameterization: action sampling supports backpropagation")

✅ SAC algorithm implementation complete (detach issue fixed)
   - Automatic entropy tuning: adaptive exploration-exploitation balance
   - Twin Q-networks: reduce Q-value overestimation
   - Soft updates: stable target network updates
   - Reparameterization: action sampling supports backpropagation


In [6]:
# 📊 Training Configuration and Environment Management

# SAC Training Configuration (using global configuration)
SAC_CONFIG = {
    # Environment parameters
    'state_dim': 4,                    # Observation dimension
    'action_dim': 4,                   # Action dimension
    'action_low': [1.0, 1.0, 5.0, 1.0],    # Action lower bounds
    'action_high': [20.0, 30.0, 10.0, 20.0],  # Action upper bounds

    # SAC hyperparameters
    'lr_actor': 3e-4,                  # Actor learning rate
    'lr_critic': 3e-4,                 # Critic learning rate
    'lr_alpha': 3e-4,                  # Entropy coefficient learning rate
    'gamma': 0.99,                     # Discount factor
    'tau': 0.005,                      # Soft update coefficient
    'alpha': 0.2,                      # Initial entropy coefficient
    'automatic_entropy_tuning': True,  # Automatic entropy tuning

    # Training parameters (using global configuration)
    'batch_size': BATCH_SIZE,                 # Batch size
    'replay_buffer_size': REPLAY_BUFFER_SIZE, # Experience buffer size
    'num_episodes': NUM_EPISODES,             # Total training episodes
    'start_steps': 500,                      # Random exploration steps
    'update_interval': 1,                     # Update interval
    'eval_interval': EVAL_INTERVAL,           # Evaluation interval
    'save_interval': SAVE_INTERVAL,           # Save interval

    # Environment parameters
    'step_interval_ns': 1_000_000_000,  # Step interval (1 second)
    'max_steps_per_episode': 500,       # Maximum steps per episode

    # Data management (using global configuration)
    'use_data_slices': True,                  # Use data slices
    'hours_per_slice': 0.167,                 # Hours per slice (10 minutes)
    'slices_per_episode': 1,                  # Slices per episode
    'data_sample_rate': DATA_SAMPLE_RATE,     # Data sampling rate
    'max_samples_per_env': 5000000,           # Maximum sample limit
}

# Create model save directories
os.makedirs('checkpoints/sac', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("✅ SAC training configuration complete (using global configuration)")
print(f"   State dimension: {SAC_CONFIG['state_dim']} (price, spread, position, PnL)")
print(f"   Action dimension: {SAC_CONFIG['action_dim']} (half_spread, skew, grid_num, grid_interval)")
print(f"   Learning rates: Actor={SAC_CONFIG['lr_actor']}, Critic={SAC_CONFIG['lr_critic']}")
print(f"   Batch size: {SAC_CONFIG['batch_size']} (global configuration)")
print(f"   Experience buffer capacity: {SAC_CONFIG['replay_buffer_size']:,} (global configuration)")
print(f"   Training episodes: {SAC_CONFIG['num_episodes']:,} (global configuration)")
print(f"   Data sampling rate: {SAC_CONFIG['data_sample_rate']*100:.0f}% (global configuration)")
print(f"   Evaluation interval: Every {SAC_CONFIG['eval_interval']} episodes")
print(f"   Save interval: Every {SAC_CONFIG['save_interval']} episodes")

✅ SAC training configuration complete (using global configuration)
   State dimension: 4 (price, spread, position, PnL)
   Action dimension: 4 (half_spread, skew, grid_num, grid_interval)
   Learning rates: Actor=0.0003, Critic=0.0003
   Batch size: 64 (global configuration)
   Experience buffer capacity: 5,000 (global configuration)
   Training episodes: 100 (global configuration)
   Data sampling rate: 10% (global configuration)
   Evaluation interval: Every 10 episodes
   Save interval: Every 10 episodes


In [7]:
# 🌍 Environment Manager (Fixed Version - Limited Maximum Sample Size)

class EnvironmentManager:
    """Manage training environments for multiple data slices (fixed version)"""

    def __init__(self, data_files, config, device):
        self.data_files = data_files
        self.config = config
        # Use passed device (supports CUDA/MPS/CPU)
        self.device = device
        self.current_idx = 0
        self.slicer = DataSlicer()
        # Get sampling rate from configuration
        self.sample_rate = config.get('data_sample_rate', 0.1)
        # Set maximum sample limit to prevent memory overflow
        self.max_samples = config.get('max_samples_per_env', 500000)  # Default 500k
        print(f"   Environment manager using device: {self.device}")
        print(f"   Data sampling rate: {self.sample_rate*100:.0f}%")
        print(f"   Maximum samples: {self.max_samples:,}")

    def create_env_from_slice(self, data_file, sample_rate=None):
        """Create environment from data slice (sampling to reduce memory)"""
        import gc

        # Use passed sampling rate or default configuration
        if sample_rate is None:
            sample_rate = self.sample_rate

        # Clear memory before loading data
        gc.collect()
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()
        elif self.device.type == 'mps':
            # MPS memory cleanup (if needed)
            torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None

        # Load data (silent mode)
        data = np.load(data_file)
        data_array = data['data']

        original_size = len(data_array)

        # Calculate sample quantity, but don't exceed maximum limit
        target_sample_size = int(original_size * sample_rate)
        actual_sample_size = min(target_sample_size, self.max_samples)

        # Sample data
        if actual_sample_size < original_size:
            indices = np.sort(np.random.choice(
                original_size, actual_sample_size, replace=False))
            data_array = data_array[indices]

        # Explicitly close file
        data.close()
        del data
        gc.collect()  # Immediate memory cleanup

        # Create HFT backtest asset
        data_asset = (
            BacktestAsset()
            .data([data_array])
            .linear_asset(1.0)
            .risk_adverse_queue_model()
            .no_partial_fill_exchange()
            .constant_latency(10_000_000, 10_000_000)
            .tick_size(TICK_SIZE)
            .lot_size(LOT_SIZE)
            .trading_value_fee_model(MAKER_FEE_RATE, TAKER_FEE_RATE)
            .power_prob_queue_model(3.0)
        )

        # Create PMM environment (using configured device)
        env = create_pmm_env(
            data_asset=data_asset,
            action_low=self.config['action_low'],
            action_high=self.config['action_high'],
            max_steps=self.config['max_steps_per_episode'],
            device=str(self.device),  # Pass device string
            risk_penalty_weight=0.01,
            step_interval_ns=self.config['step_interval_ns'],
        )

        return env

    def get_next_env(self):
        """Get next training environment (cyclically use slices)"""
        data_file = self.data_files[self.current_idx]
        env = self.create_env_from_slice(data_file)  # Use configured sampling rate
        self.current_idx = (self.current_idx + 1) % len(self.data_files)
        return env, data_file

    def get_random_env(self):
        """Randomly get a training environment"""
        data_file = random.choice(self.data_files)
        # Reset current_idx to maintain log consistency
        self.current_idx = self.data_files.index(data_file)
        env = self.create_env_from_slice(data_file)
        return env, data_file

    def estimate_steps(self, data_file):
        """Estimate environment steps"""
        # Consider sampling rate and maximum sample count
        try:
            data = np.load(data_file)
            original_size = len(data['data'])
            data.close()

            # Calculate actual samples used
            target_samples = int(original_size * self.sample_rate)
            actual_samples = min(target_samples, self.max_samples)

            # Estimate steps
            return int(actual_samples / (self.config['step_interval_ns'] / 1e9))
        except:
            return 100  # Default estimate


# Use previously obtained data slices (from Cell 4)
print("📊 Using prepared data slices...")
if 'data_files' not in globals():
    print("❌ Error: Data slices not prepared! Please execute Cell 4 first")
    raise NameError("data_files not defined, please execute data configuration cell first")

# Directly use existing data_files variable
data_slices = data_files

print(f"✅ Using {len(data_slices)} data slices")
print(f"   Duration per slice: {SAC_CONFIG['hours_per_slice']} hours")
print(f"   Data sampling rate: {SAC_CONFIG.get('data_sample_rate', 0.1)*100:.0f}%")
print(f"   Maximum samples: {SAC_CONFIG.get('max_samples_per_env', 500000):,}")
print(
    f"   Training mode: {'Sequential' if SAC_CONFIG['slices_per_episode'] == 1 else 'Random'} slice usage")

# Create environment manager
env_manager = EnvironmentManager(data_slices, SAC_CONFIG, device)
print("✅ Environment manager created successfully (supports CUDA/MPS/CPU)")

📊 Using prepared data slices...
✅ Using 144 data slices
   Duration per slice: 0.167 hours
   Data sampling rate: 10%
   Maximum samples: 5,000,000
   Training mode: Sequential slice usage
   Environment manager using device: mps
   Data sampling rate: 10%
   Maximum samples: 5,000,000
✅ Environment manager created successfully (supports CUDA/MPS/CPU)


In [8]:
# 🚀 SAC Training Loop (Enhanced Version)

def train_sac():
    """SAC main training loop (enhanced version)"""
    import gc

    # Initialize SAC algorithm (using global device variable)
    sac = SAC(
        state_dim=SAC_CONFIG['state_dim'],
        action_dim=SAC_CONFIG['action_dim'],
        action_low=SAC_CONFIG['action_low'],
        action_high=SAC_CONFIG['action_high'],
        device=device,  # Use selected device (CUDA/MPS/CPU)
        lr_actor=SAC_CONFIG['lr_actor'],
        lr_critic=SAC_CONFIG['lr_critic'],
        lr_alpha=SAC_CONFIG['lr_alpha'],
        gamma=SAC_CONFIG['gamma'],
        tau=SAC_CONFIG['tau'],
        alpha=SAC_CONFIG['alpha'],
        automatic_entropy_tuning=SAC_CONFIG['automatic_entropy_tuning']
    )

    # Initialize experience replay buffer
    replay_buffer = ReplayBuffer(SAC_CONFIG['replay_buffer_size'])

    # Training statistics
    episode_rewards = []
    episode_steps = []
    training_losses = []
    total_steps = 0
    best_reward = -float('inf')

    print(f"\n🎯 Starting SAC training (enhanced version)")
    print(f"   Device: {device}")
    print(f"   Total episodes: {SAC_CONFIG['num_episodes']:,}")
    print(f"   Random exploration steps: {SAC_CONFIG['start_steps']:,}")
    print(f"   Batch size: {SAC_CONFIG['batch_size']}")
    print(f"   Data sampling rate: {SAC_CONFIG['data_sample_rate']*100:.0f}%")
    print(f"   Maximum steps per episode: {SAC_CONFIG['max_steps_per_episode']}")

    # Training loop
    for episode in tqdm(range(SAC_CONFIG['num_episodes']), desc="Training Progress"):
        try:
            # Periodic memory cleanup
            if episode % 10 == 0:
                gc.collect()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
                elif device.type == 'mps':
                    torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None

            # Get new environment (using different data slices)
            if episode % 100 == 0:  # Show detailed info every 100 episodes
                print(f"\n📌 Episode {episode+1}/{SAC_CONFIG['num_episodes']}")

            env, data_file = env_manager.get_next_env()
            estimated_steps = env_manager.estimate_steps(data_file)

            # Reset environment
            tensordict = env.reset()
            state = tensordict['observation'].cpu().numpy()

            episode_reward = 0
            episode_step = 0

            # Episode loop
            done = False
            while not done and episode_step < SAC_CONFIG['max_steps_per_episode']:
                # Select action
                if total_steps < SAC_CONFIG['start_steps']:
                    # Random exploration
                    action = np.random.uniform(
                        SAC_CONFIG['action_low'],
                        SAC_CONFIG['action_high']
                    )
                    # Ensure integer parameters
                    action[0] = round(action[0])
                    action[1] = round(action[1])
                    action[2] = round(action[2])
                    action[3] = round(action[3])
                else:
                    # SAC policy action selection
                    action = sac.select_action(state, evaluate=False)

                # Execute action (using configured device)
                action_tensor = torch.tensor(
                    action, dtype=torch.float32, device=device)
                action_td = TensorDict(
                    {"action": action_tensor}, batch_size=(), device=device)

                try:
                    next_tensordict = env.step(action_td)
                except Exception as e:
                    if episode % 100 == 0:  # Reduce error output frequency
                        print(f"   ⚠️ Step error: {e}")
                    done = True
                    break

                if 'next' in next_tensordict:
                    next_state = next_tensordict['next']['observation'].cpu(
                    ).numpy()
                    reward = next_tensordict['next']['reward'].cpu().item()
                    done = next_tensordict['next']['done'].cpu().item() > 0.5

                    # Store experience
                    replay_buffer.push(state, action, reward, next_state, done)

                    # Update state
                    state = next_state
                    episode_reward += reward
                    episode_step += 1
                    total_steps += 1

                    # Update networks
                    if total_steps >= SAC_CONFIG['start_steps'] and \
                       total_steps % SAC_CONFIG['update_interval'] == 0 and \
                       len(replay_buffer) >= SAC_CONFIG['batch_size']:
                        losses = sac.update(
                            replay_buffer, SAC_CONFIG['batch_size'])
                        if losses:
                            training_losses.append(losses)
                else:
                    done = True

            # Record episode statistics
            episode_rewards.append(episode_reward)
            episode_steps.append(episode_step)

            # Update best reward
            if episode_reward > best_reward:
                best_reward = episode_reward
                # Save best model
                best_model_path = "checkpoints/sac/sac_model_best.pth"
                sac.save(best_model_path)

            # Clean up environment
            env.close()
            del env

        except Exception as e:
            if episode % 100 == 0:  # Reduce error output frequency
                print(f"   ❌ Episode {episode+1} error: {e}")
            continue

        # Periodic evaluation and saving
        if (episode + 1) % SAC_CONFIG['eval_interval'] == 0:
            if episode_rewards:
                recent_rewards = episode_rewards[-min(
                    SAC_CONFIG['eval_interval'], len(episode_rewards)):]
                avg_reward = np.mean(recent_rewards)
                avg_steps = np.mean(
                    episode_steps[-min(SAC_CONFIG['eval_interval'], len(episode_steps)):])

                print(
                    f"\n📊 Episode {episode + 1}/{SAC_CONFIG['num_episodes']}")
                print(f"   Average reward: {avg_reward:.4f}")
                print(f"   Best reward: {best_reward:.4f}")
                print(f"   Average steps: {avg_steps:.0f}")
                print(f"   Total steps: {total_steps:,}")
                print(f"   Buffer size: {len(replay_buffer):,}")

                # Show recent losses
                if training_losses and len(training_losses) > 0:
                    recent_losses = training_losses[-min(
                        100, len(training_losses)):]
                    avg_critic_loss = np.mean(
                        [l['critic_loss'] for l in recent_losses])
                    avg_actor_loss = np.mean(
                        [l['actor_loss'] for l in recent_losses])
                    avg_alpha = np.mean([l['alpha'] for l in recent_losses])
                    print(f"   Average Critic loss: {avg_critic_loss:.6f}")
                    print(f"   Average Actor loss: {avg_actor_loss:.6f}")
                    print(f"   Current entropy coefficient α: {avg_alpha:.4f}")

        # Periodic model saving
        if (episode + 1) % SAC_CONFIG['save_interval'] == 0:
            model_path = f"checkpoints/sac/sac_model_episode_{episode + 1}.pth"
            sac.save(model_path)
            print(f"💾 Model saved: {model_path}")

            # Save training progress
            progress = {
                'episode': episode + 1,
                'total_steps': total_steps,
                'best_reward': best_reward,
                'recent_rewards': episode_rewards[-100:] if len(episode_rewards) > 100 else episode_rewards,
            }
            progress_path = f"checkpoints/sac/training_progress.json"
            with open(progress_path, 'w') as f:
                json.dump(progress, f, indent=2)

    # Save final model
    if episode_rewards:
        final_model_path = "checkpoints/sac/sac_model_final.pth"
        sac.save(final_model_path)
        print(f"\n✅ Training complete! Final model saved: {final_model_path}")

        # Save complete training statistics
        stats = {
            'episode_rewards': episode_rewards,
            'episode_steps': episode_steps,
            'total_episodes': len(episode_rewards),
            'total_steps': total_steps,
            'best_reward': best_reward,
            'config': SAC_CONFIG
        }

        stats_path = f"logs/training_stats_{time.strftime('%Y%m%d_%H%M%S')}.json"
        with open(stats_path, 'w') as f:
            json.dump(stats, f, indent=2)
        print(f"📊 Training statistics saved: {stats_path}")

    return sac, episode_rewards


print("✅ SAC training system ready (enhanced version)")
print(f"   - Using {len(data_slices)} data slices in rotation for training")
print(f"   - Data sampling {SAC_CONFIG['data_sample_rate']*100:.0f}%")
print(f"   - Supporting device acceleration: {device}")
print(f"   - Enhanced error handling and memory management")
print(f"   - Saves best model and training progress")

✅ SAC training system ready (enhanced version)
   - Using 144 data slices in rotation for training
   - Data sampling 10%
   - Supporting device acceleration: mps
   - Enhanced error handling and memory management
   - Saves best model and training progress


In [9]:
# 📈 Execute SAC Training

print("🎯 Preparing to start SAC training...")
print(f"   Device: {device}")
print(f"   Data slices: {len(data_slices)} slices")

print("\n📊 Current configuration (from global settings):")
print(f"   Training episodes: {SAC_CONFIG['num_episodes']:,}")
print(f"   Batch size: {SAC_CONFIG['batch_size']}")
print(f"   Experience buffer: {SAC_CONFIG['replay_buffer_size']:,}")
print(f"   Data sampling rate: {SAC_CONFIG['data_sample_rate']*100:.0f}%")
print(f"   Max steps per episode: {SAC_CONFIG['max_steps_per_episode']}")
print(f"   Save interval: Every {SAC_CONFIG['save_interval']} episodes")
print(f"   Evaluation interval: Every {SAC_CONFIG['eval_interval']} episodes")

# Show expectations based on training episodes
if SAC_CONFIG['num_episodes'] <= 20:
    print(f"\n🧪 Quick test mode: {SAC_CONFIG['num_episodes']} episodes")
    print("   Expected time: 5-10 minutes")
    print("   Purpose: Validate environment configuration")
elif SAC_CONFIG['num_episodes'] <= 100:
    print(f"\n⚡ Fast training mode: {SAC_CONFIG['num_episodes']} episodes")
    print("   Expected time: 30-60 minutes")
    print("   Purpose: Quick prototype validation")
elif SAC_CONFIG['num_episodes'] <= 500:
    print(f"\n💪 Standard training mode: {SAC_CONFIG['num_episodes']} episodes")
    print("   Expected time: 4-6 hours")
    print("   Purpose: Obtain usable strategy")
else:
    print(f"\n🔥 Deep training mode: {SAC_CONFIG['num_episodes']} episodes")
    print("   Expected time: 8-12 hours")
    print("   Purpose: Achieve best performance")
    print("   Suggestion: Let program run in background")

print("\n💡 Tip: Adjust NUM_EPISODES in the first configuration cell")

# Pre-training preparation
print("\n📝 Pre-training checklist:")
print("   ✓ Configuration parameters set (first cell)")
print("   ✓ All prerequisite cells executed")
print("   ✓ Data slices prepared")
print("   ✓ Sufficient memory (recommend >8GB)")

try:
    print("\n✅ Environment ready, starting training...")
    print("-" * 50)

    # Clean memory
    import gc
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    elif device.type == 'mps':
        torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None

    # Record start time
    start_time = time.time()

    # Execute training
    sac_agent, episode_rewards = train_sac()

    # Calculate training time
    training_time = time.time() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = int(training_time % 60)

    # Display results
    print("\n🎉 Training complete!")
    print(f"   Training time: {hours} hours {minutes} minutes {seconds} seconds")

    if episode_rewards:
        print(f"\n📊 Training statistics:")
        print(f"   Completed episodes: {len(episode_rewards)}")
        print(f"   Average reward: {np.mean(episode_rewards):.4f}")
        print(f"   Standard deviation: {np.std(episode_rewards):.4f}")

        # Show statistics for the last portion
        if len(episode_rewards) > 10:
            recent_n = min(50, len(episode_rewards))
            recent = episode_rewards[-recent_n:]
            print(f"\n   Last {recent_n} episodes statistics:")
            print(f"   Average: {np.mean(recent):.4f}")
            print(f"   Best: {max(recent):.4f}")
            print(f"   Worst: {min(recent):.4f}")

        # Overall statistics
        print(f"\n   Overall statistics:")
        print(f"   Best reward: {max(episode_rewards):.4f}")
        print(f"   Worst reward: {min(episode_rewards):.4f}")
        success = len([r for r in episode_rewards if r > 0])
        print(f"   Success rate: {success}/{len(episode_rewards)} ({success/len(episode_rewards)*100:.1f}%)")

    print("\n📁 Saved files:")
    print("   - Final model: checkpoints/sac/sac_model_final.pth")
    print("   - Best model: checkpoints/sac/sac_model_best.pth")
    print("   - Training statistics: logs/training_stats_*.json")
    print("   - Training progress: checkpoints/sac/training_progress.json")

    print("\nNext steps:")
    print("1. Run the next cell for model evaluation")
    print("2. Adjust training parameters (first configuration cell)")
    print("3. View training curves and statistical analysis")

except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
    print("Tips:")
    print("- Model automatically saved to latest checkpoint")
    print("- Can resume training from checkpoint")
    print("- Check checkpoints/sac/training_progress.json for progress")

except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nSuggested solutions:")
    print("1. Check if memory is sufficient")
    print("2. Reduce BATCH_SIZE and REPLAY_BUFFER_SIZE in configuration cell")
    print("3. Reduce DATA_SAMPLE_RATE in configuration cell")
    print("4. Restart kernel and re-execute")
    import traceback
    traceback.print_exc()

🎯 Preparing to start SAC training...
   Device: mps
   Data slices: 144 slices

📊 Current configuration (from global settings):
   Training episodes: 100
   Batch size: 64
   Experience buffer: 5,000
   Data sampling rate: 10%
   Max steps per episode: 500
   Save interval: Every 10 episodes
   Evaluation interval: Every 10 episodes

⚡ Fast training mode: 100 episodes
   Expected time: 30-60 minutes
   Purpose: Quick prototype validation

💡 Tip: Adjust NUM_EPISODES in the first configuration cell

📝 Pre-training checklist:
   ✓ Configuration parameters set (first cell)
   ✓ All prerequisite cells executed
   ✓ Data slices prepared
   ✓ Sufficient memory (recommend >8GB)

✅ Environment ready, starting training...
--------------------------------------------------

🎯 Starting SAC training (enhanced version)
   Device: mps
   Total episodes: 100
   Random exploration steps: 500
   Batch size: 64
   Data sampling rate: 10%
   Maximum steps per episode: 500


Training Progress:   0%|          | 0/100 [00:00<?, ?it/s]


📌 Episode 1/100

📊 Episode 10/100
   Average reward: -9.7322
   Best reward: 202.9004
   Average steps: 347
   Total steps: 3,473
   Buffer size: 3,473
   Average Critic loss: 109.136588
   Average Actor loss: -8.321618
   Current entropy coefficient α: 0.4215
💾 Model saved: checkpoints/sac/sac_model_episode_10.pth

📊 Episode 20/100
   Average reward: 15.0802
   Best reward: 202.9004
   Average steps: 399
   Total steps: 7,463
   Buffer size: 5,000
   Average Critic loss: 46.368700
   Average Actor loss: -15.087455
   Current entropy coefficient α: 0.1317
💾 Model saved: checkpoints/sac/sac_model_episode_20.pth

📊 Episode 30/100
   Average reward: 89.8985
   Best reward: 348.6184
   Average steps: 390
   Total steps: 11,368
   Buffer size: 5,000
   Average Critic loss: 19.527171
   Average Actor loss: 14.999530
   Current entropy coefficient α: 0.0434
💾 Model saved: checkpoints/sac/sac_model_episode_30.pth

📊 Episode 40/100
   Average reward: 39.9550
   Best reward: 348.6184
   Average

In [10]:
# 📊 Model Evaluation and Analysis

def evaluate_sac_agent(sac_agent, env_manager, num_eval_episodes=10):
    """Evaluate trained SAC agent"""

    print(f"\n🔍 Evaluating SAC agent performance...")
    print(f"   Evaluation episodes: {num_eval_episodes}")

    eval_rewards = []
    eval_actions = []
    eval_pnls = []

    for episode in tqdm(range(num_eval_episodes), desc="Evaluation Progress"):
        # Get evaluation environment
        env, data_file = env_manager.get_random_env()

        # Reset environment
        tensordict = env.reset()
        state = tensordict['observation'].cpu().numpy()

        episode_reward = 0
        episode_actions = []
        done = False
        steps = 0

        while not done and steps < SAC_CONFIG['max_steps_per_episode']:
            # Use deterministic policy (evaluation mode)
            action = sac_agent.select_action(state, evaluate=True)
            episode_actions.append(action.tolist())

            # Execute action
            action_tensor = torch.tensor(
                action, dtype=torch.float32, device=device)
            action_td = TensorDict(
                {"action": action_tensor}, batch_size=(), device=device)
            next_tensordict = env.step(action_td)

            if 'next' in next_tensordict:
                state = next_tensordict['next']['observation'].cpu().numpy()
                reward = next_tensordict['next']['reward'].cpu().item()
                done = next_tensordict['next']['done'].cpu().item() > 0.5

                episode_reward += reward
                steps += 1

                # Record final PnL
                if done or steps >= SAC_CONFIG['max_steps_per_episode'] - 1:
                    strategy_state = env._get_strategy_state()
                    eval_pnls.append(strategy_state['pnl'])
            else:
                done = True

        eval_rewards.append(episode_reward)
        eval_actions.append(episode_actions)

        # Clean up environment
        del env

    # Statistical analysis
    avg_reward = np.mean(eval_rewards)
    std_reward = np.std(eval_rewards)
    avg_pnl = np.mean(eval_pnls)
    success_rate = len([r for r in eval_rewards if r > 0]
                       ) / len(eval_rewards) * 100

    print(f"\n📊 Evaluation results:")
    print(f"   Average reward: {avg_reward:.4f} ± {std_reward:.4f}")
    print(f"   Average PnL: ${avg_pnl:.2f}")
    print(f"   Success rate: {success_rate:.1f}%")
    print(f"   Best reward: {max(eval_rewards):.4f}")
    print(f"   Worst reward: {min(eval_rewards):.4f}")

    # Analyze most commonly used action parameters (note: these parameters are all integers in actual use)
    if eval_actions:
        all_actions = [
            action for episode in eval_actions for action in episode]
        actions_array = np.array(all_actions)

        # Calculate statistics
        avg_params = np.mean(actions_array, axis=0)
        std_params = np.std(actions_array, axis=0)

        # Calculate most common values (mode)
        from scipy import stats
        mode_params = []
        for i in range(4):
            mode_result = stats.mode(
                actions_array[:, i].astype(int), keepdims=True)
            mode_params.append(mode_result.mode[0])

        print(f"\n🎯 Learned strategy parameters (integer parameters):")
        print(f"   Note: The following shows statistical averages, actual execution uses integers")
        print(f"   {'Parameter':<15} {'Average':<15} {'Std Dev':<10} {'Most Common':<10}")
        print(f"   {'-'*50}")

        param_names = ['Half Spread (ticks)', 'Skew Factor (ticks)', 'Grid Layers', 'Grid Interval (ticks)']
        for i, name in enumerate(param_names):
            print(
                f"   {name:<15} {avg_params[i]:>6.1f} ± {std_params[i]:<8.1f} {int(mode_params[i]):>10d}")

        print(f"\n   📌 Actual usage parameters (after rounding):")
        actual_params = np.round(avg_params).astype(int)
        print(f"   Half spread: {actual_params[0]} ticks")
        print(f"   Skew factor: {actual_params[1]} ticks")
        print(f"   Grid layers: {actual_params[2]} layers")
        print(f"   Grid interval: {actual_params[3]} ticks")

    return {
        'rewards': eval_rewards,
        'pnls': eval_pnls,
        'actions': eval_actions,
        'avg_reward': avg_reward,
        'avg_pnl': avg_pnl,
        'success_rate': success_rate
    }


# If training is complete, perform evaluation
if 'sac_agent' in locals():
    eval_results = evaluate_sac_agent(
        sac_agent, env_manager, num_eval_episodes=10)

    # Save evaluation results
    eval_path = f"logs/eval_results_{time.strftime('%Y%m%d_%H%M%S')}.json"
    with open(eval_path, 'w') as f:
        json.dump({
            'avg_reward': eval_results['avg_reward'],
            'avg_pnl': eval_results['avg_pnl'],
            'success_rate': eval_results['success_rate'],
            'rewards': eval_results['rewards'],
            'pnls': eval_results['pnls']
        }, f, indent=2)
    print(f"\n💾 Evaluation results saved: {eval_path}")
else:
    print("⚠️ Please run the training cell first")


🔍 Evaluating SAC agent performance...
   Evaluation episodes: 10


Evaluation Progress:   0%|          | 0/10 [00:00<?, ?it/s]


📊 Evaluation results:
   Average reward: 146.2028 ± 161.1587
   Average PnL: $144.98
   Success rate: 60.0%
   Best reward: 500.1405
   Worst reward: -42.0186

🎯 Learned strategy parameters (integer parameters):
   Note: The following shows statistical averages, actual execution uses integers
   Parameter       Average         Std Dev    Most Common
   --------------------------------------------------
   Half Spread (ticks)   19.8 ± 0.6              20
   Skew Factor (ticks)    2.7 ± 5.8               1
   Grid Layers        5.5 ± 1.1               5
   Grid Interval (ticks)   11.3 ± 9.2               1

   📌 Actual usage parameters (after rounding):
   Half spread: 20 ticks
   Skew factor: 3 ticks
   Grid layers: 5 layers
   Grid interval: 11 ticks

💾 Evaluation results saved: logs/eval_results_20250812_083256.json


## 📋 Summary

This notebook successfully implements a PMM market making strategy reinforcement learning training system based on the **SAC (Soft Actor-Critic)** algorithm.

### ✅ Implemented Features

1. **Complete SAC Algorithm**

    - Actor Network: Policy network, outputs action distribution
    - Critic Network: Twin Q-network architecture, reduces Q-value overestimation
    - Automatic Entropy Tuning: Balances exploration and exploitation
    - Experience Replay: Improves sample efficiency

2. **Data Slicing System**

    - Solves hftbacktest's lack of time-jumping support
    - Uses different data slices for each episode
    - Avoids overfitting to single market states

3. **Environment Management**

    - Automatically creates and manages multiple trading environments
    - Supports sequential or random data slice usage
    - Efficient resource management

4. **Training and Evaluation**
    - Complete training loop
    - Periodic model and statistics saving
    - Evaluation system to test learning effectiveness

### 🎯 Comparison with Random Search

| Feature | Random Search (Original) | SAC (New Implementation) |
| ------- | ----------------------- | ------------------------ |
| Learning Capability | ❌ No learning, purely random | ✅ Continuous learning optimization |
| Sample Efficiency | ❌ Independent sampling each time | ✅ Experience replay reuse |
| Convergence | ❌ No convergence guarantee | ✅ Theoretical convergence guarantee |
| Exploration Strategy | ❌ Completely random | ✅ Intelligent exploration (entropy regularization) |
| Adaptability | ❌ Cannot adapt | ✅ Adapts to market changes |

### 🚀 Future Optimization Suggestions

1. **Network Architecture Optimization**

    - Use LSTM/GRU to process temporal information
    - Add attention mechanisms
    - Increase network depth and width

2. **Feature Engineering**

    - Add more market microstructure features
    - Technical indicators (MA, RSI, etc.)
    - Order book depth information

3. **Training Optimization**

    - Implement Prioritized Experience Replay (PER)
    - Add distributed training support
    - Use larger batch sizes

4. **Strategy Improvements**
    - Multi-asset joint training
    - Risk-adjusted reward functions
    - Add stop-loss and risk control mechanisms

### 💡 Usage Recommendations

1. **Adjust Training Episodes**: Adjust `num_episodes` based on computational resources
2. **Monitor Training Process**: Observe loss curves and reward trends
3. **Parameter Tuning**: Use grid search or Bayesian optimization to tune hyperparameters
4. **Validate Generalization**: Test model on unseen data

### 📊 Performance Metrics

After training completion, the strategy can be evaluated using the following metrics:

-   **Average Reward**: Measures overall strategy performance
-   **Success Rate**: Proportion of profitable episodes
-   **Sharpe Ratio**: Risk-adjusted returns
-   **Maximum Drawdown**: Risk control capability